# 02_EDA on Supply Chain Data to see what should be cleaned

In [0]:
# Load the raw supply chain data from the bronze layer table
df = spark.sql("SELECT * FROM supply_chain_demo.bronze.raw_supply_chain")

# Display the dataframe in a table format
display(df)

- Want to change to snakcase

In [0]:
# Load metadata table that contains field descriptions
df_metadata = spark.sql("FROM supply_chain_demo.bronze.metadata")

# Display the metadata table
df_metadata.display()

In [0]:
# Count total rows in metadata and total columns in main dataframe
# This helps verify if metadata has info for all columns
df_metadata.count(), len(df.columns)

In [0]:
# Print all column names in the dataframe to see what fields we have
print(df.columns)

In [0]:
# Show all the field names listed in the metadata table
df_metadata.select("FIELDS").display()

In [0]:
# Compare fields: find columns that exist in one dataset but not the other
# Get field names from metadata and compare with actual dataframe columns
set(df_metadata.select("FIELDS").toPandas()["FIELDS"]).symmetric_difference(
    set(df.columns)
)

In [0]:
# Count how many unique/distinct zip codes exist in the Order Zipcode column
df.select("Order Zipcode").distinct().count()

In [0]:
# Print the total number of rows and columns in the dataset
print(f"Number of rows {df.count()}")
print(f"Number of columns {len(df.columns)}")

# Print the schema (data types) of all columns in the dataframe
df.printSchema()

In [0]:
# Show all distinct/unique transaction types in the dataset
df.select("Type").distinct().display()

In [0]:
# Display the full dataframe in table format
df.display()

In [0]:
df.select("Type").distinct().display()

In [0]:
# Generate summary statistics (count, mean, min, max, etc.) for all numeric columns
# Filter to include only numeric data types: int, bigint, double, decimal
df.select(
    [
        column
        for column, type_ in df.dtypes
        if type_ in ("int", "bigint", "double", "decimal")
    ]
).summary().display()

In [0]:
# Explore distinct values in customer and order location columns
# Check what countries appear in Customer Country
df.select("Customer Country").distinct().display()

# Check what countries appear in Order Country
df.select("Order Country").distinct().display()

# Check customer password values (to see if data is masked)
df.select("Customer Password").distinct().display()

# Check distinct customer zip codes
df.select("Customer Zipcode").distinct().display()

In [0]:
# Count how many null (missing) values exist in Customer Zipcode column
df.select("Customer Zipcode").filter(df["Customer Zipcode"].isNull()).count()

In [0]:
# Count null values across ALL columns in the dataframe
from pyspark.sql.functions import col, sum as spark_sum

# For each column, count how many rows have null values
null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

# Convert results to a dictionary and filter to show only columns with nulls
null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

In [0]:
# Display the shipping date column to inspect its format
df.select("shipping date (DateOrders)").display()

In [0]:
# Show all distinct/unique product descriptions in the dataset
df.select("Product Description").distinct().display()

In [0]:
# Display the full dataframe again for review
display(df)

# Exercise
## Summary of cleaning steps
We note down what to be cleaned based on looking at the data, reading metadata and doing some EDA. This list is not comprehensive, I don't want to take away all the fun for you :)

## Drop
Customer Email (data is masked)
Customer Password
Product Description (all are nulls)

## Rounding errors
Order Item Product Price -> 2 decimals
Order Item Product Price -> 2 decimals
Benefit per order -> 2 decimals
Sales per customer -> 2 decimals ...

## Customer country
EE. UU. -> United States as it is the spanish abbreviation

## Nulls
fill Customer Lname with missing
fill Customer Zipcode with missing
fill Order Zipcode with missing

## Data type
shipping date - string -> Timestamp

## Derived columns 
will not remove them in silver layer, but will compute them in Gold layer instead

## Order Item Total 
derived from (Order Item Quantity)(Order Item Product Price)(1-Order Item Discount Rate), calculate and test this ...

## Rename columns
use snake_case convention instead (this is a design choice)

In [0]:
# Convert all column names from "Title Case With Spaces" to "snake_case" format
import re 

def to_snake_case(name):
    # Replace spaces with underscores and convert to lowercase
    return re.sub(r"[\s]+", "_", name.strip().casefold())

def rename_columns_to_snake_case(df):
    # Apply snake_case conversion to all column names
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

# Create a new dataframe with renamed columns
df_column_alias = rename_columns_to_snake_case(df)

# Display first 5 rows to verify the new column names
df_column_alias.limit(5).display()

In [0]:
# Convert shipping date from string to proper timestamp format
from pyspark.sql.functions import to_timestamp, col 

# Parse the date string using the format "M/d/yyyy H:mm" and convert to timestamp
df_timestamp = df_column_alias.withColumn(
    "shipping_date", to_timestamp(col("shipping_date_(dateorders)"), "M/d/yyyy H:mm")
)

# Display first 3 shipping dates to verify the conversion
df_timestamp.select("shipping_date").limit(3).display()

In [0]:
# Apply all data cleaning transformations
from pyspark.sql.functions import coalesce, lit, when

df_cleaned = (
    df_timestamp
    # Fill null last names with "-" placeholder
    .withColumn("customer_lname", coalesce("customer_lname", lit("-")))
    # Fill null customer zip codes with "unknown" (convert to string first)
    .withColumn(
        "customer_zipcode",
        coalesce(col("customer_zipcode").cast("string"), lit("unknown")),
    )
    # Fill null order zip codes with "unknown" (convert to string first)
    .withColumn(
        "order_zipcode", coalesce(col("order_zipcode").cast("string"), lit("unknown"))
    )
    # Replace Spanish abbreviation "EE. UU." with "United States"
    .withColumn(
        "customer_country",
        when(col("customer_country") == "EE. UU.", "United States").otherwise(
            col("customer_country")
        ),
    )
    # Drop columns that are not useful: product_description (all nulls), customer_email (masked), customer_password
).drop("product_description", "customer_email", "customer_password")

# Display the cleaned dataframe
display(df_cleaned)

In [0]:
# Display the final cleaned dataframe
df_cleaned.display()